# Experiments Notebook\n
Use this notebook for prototyping and testing.

In [2]:
# Import libraries for PDF and directory document loading (LangChain), environment configuration,
# core Python utilities, ChromaDB vector storage, OpenAI models, and recursive text splitting.

# PDF + directory document loading (LangChain)
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader, PyPDFDirectoryLoader
from langchain_core.documents import Document

# Prompt templates
from langchain_core.prompts import ChatPromptTemplate

# LCEL — chain building blocks
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Token usage tracking
from langchain_community.callbacks import get_openai_callback

# Recursive text splitting
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Environment configuration
from dotenv import load_dotenv
import os

# Core Python utilities
import json
import uuid
import shutil
from pathlib import Path
from typing import List, Dict, Any

# ChromaDB vector storage (LangChain integration)
from langchain_community.vectorstores import Chroma

# OpenAI models (LangChain)
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# Load environment variables from .env
load_dotenv()


True

In [3]:
# Define simple, easy-to-understand dynamic functions (get_pdf_files, enrich_metadata, load_pdfs)
# to retrieve all PDFs from the specified directory using PyPDFDirectoryLoader,
# enrich each document with structured metadata fields,
# and load them into LangChain Document objects with print-based status messages.

data_path = r"C:\Users\ROHIT\Work\LLMOPS\Document_Portal\data\document_analyzer"


def get_pdf_files(directory: str) -> list:
    """Return a sorted list of all PDF file paths found in the directory."""
    pdf_files = list(Path(directory).glob("**/*.pdf"))
    print(f"[get_pdf_files] Directory : {directory}")
    print(f"[get_pdf_files] PDFs found: {len(pdf_files)}")
    return sorted(pdf_files)


def enrich_metadata(documents: List[Document]) -> List[Document]:
    """Add structured metadata fields to every document:
       source, file_name, date, author, section, page, document_type, access_level.
    """
    for doc in documents:
        m = doc.metadata

        # source — full file path (already set by loader, ensure it's a string)
        source = m.get("source", "")
        m["source"] = str(source)

        # file_name — just the filename from the source path
        m["file_name"] = Path(source).name if source else "unknown"

        # date — extract YYYY-MM-DD from creationdate if available
        raw_date = m.get("creationdate", "")
        m["date"] = raw_date[:10] if raw_date else "unknown"

        # author — from PDF metadata, fallback to unknown
        m["author"] = m.get("author", "").strip() or "unknown"

        # section — use page_label if available, else "unknown"
        m["section"] = str(m.get("page_label", "")).strip() or "unknown"

        # page — already set by loader as int; keep it
        m["page"] = m.get("page", 0)

        # document_type — all files here are research papers
        m["document_type"] = "research_paper"

        # access_level — default to public
        m["access_level"] = "public"

    print(f"[enrich_metadata] Enriched {len(documents)} document(s).")
    return documents


def load_pdfs(directory: str) -> List[Document]:
    """Load all PDFs from a directory and enrich their metadata."""
    pdf_files = get_pdf_files(directory)
    if not pdf_files:
        print("[load_pdfs] No PDFs found — returning empty list.")
        return []

    print(f"[load_pdfs] Loading {len(pdf_files)} PDF(s) with PyPDFDirectoryLoader...")
    loader = PyPDFDirectoryLoader(directory, recursive=True, silent_errors=True)
    documents = loader.load()
    print(f"[load_pdfs] Loaded {len(documents)} raw page document(s).")

    documents = enrich_metadata(documents)

    print(f"[load_pdfs] Done. Total pages loaded: {len(documents)}")
    return documents


In [4]:
documents = load_pdfs(data_path)
documents

[get_pdf_files] Directory : C:\Users\ROHIT\Work\LLMOPS\Document_Portal\data\document_analyzer
[get_pdf_files] PDFs found: 8
[load_pdfs] Loading 8 PDF(s) with PyPDFDirectoryLoader...
[load_pdfs] Loaded 240 raw page document(s).
[enrich_metadata] Enriched 240 document(s).
[load_pdfs] Done. Total pages loaded: 240


[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-05-29T00:24:28+00:00', 'author': 'unknown', 'keywords': '', 'moddate': '2025-05-29T00:24:28+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'C:\\Users\\ROHIT\\Work\\LLMOPS\\Document_Portal\\data\\document_analyzer\\AI_Agents_vs_Agentic_AI_A_Conceptual.pdf', 'total_pages': 36, 'page': 0, 'page_label': '1', 'file_name': 'AI_Agents_vs_Agentic_AI_A_Conceptual.pdf', 'date': '2025-05-29', 'section': '1', 'document_type': 'research_paper', 'access_level': 'public'}, page_content='AI Agents vs. Agentic AI: A Conceptual\nTaxonomy, Applications and Challenges\nRanjan Sapkota∗‡, Konstantinos I. Roumeliotis †, Manoj Karkee ∗‡\n∗Cornell University, Department of Biological and Environmental Engineering, USA\n†University of the Peloponnese, Department of Informatics and